# `wpt_name` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `high-cardinality-category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'wpt_name'
feature_metadata = {'order': 8, 'name': 'wpt_name', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'exclude raw exact value from the first baseline', 'finding': 'Most levels are sparse and more than half of test rows use unseen exact names.', 'decision': 'Exclude raw one-hot names initially; test cross-fitted frequency or text features separately.', 'risk': 'In-sample lookup performance is dominated by memorisation and geographic proxying.', 'sentinel_tokens': ['None', 'none', 'unknown', 'not known'], 'related': [{'feature': 'subvillage', 'reason': 'Waterpoint names may repeat within local settlements.'}, {'feature': 'ward', 'reason': 'Administrative context can disambiguate generic waterpoint names.'}, {'feature': 'scheme_name', 'reason': 'Waterpoint and scheme names may share project identity.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for wpt_name.


## Supported target evidence


In [2]:
sentinel_tokens = ['None', 'none', 'unknown', 'not known']
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
wpt_name,,,,,
none,3565,True,73.77,2.13,24.10
shuleni,1748,True,49.14,8.18,42.68
zahanati,830,True,51.81,9.40,38.80
msikitini,535,True,49.16,8.22,42.62
kanisani,323,True,47.99,6.50,45.51
bombani,271,True,58.30,7.75,33.95
sokoni,260,True,46.15,10.00,43.85
ofisini,254,True,43.31,4.33,52.36
school,208,True,41.83,6.25,51.92


status_group,rows,non functional (%)
wpt_name,,
ofisini,254,52.36
school,208,51.92
kanisani,323,45.51
sokoni,260,43.85
shuleni,1748,42.68
msikitini,535,42.62
madukani,104,42.31
zahanati,830,38.80
shule ya msingi,199,36.68


## Observation

Most levels are sparse and more than half of test rows use unseen exact names.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Exclude raw one-hot names initially; test cross-fitted frequency or text features separately.

**Risk to carry forward:** In-sample lookup performance is dominated by memorisation and geographic proxying.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
wpt_name,candidate,exclude raw exact value from the first baseline,Most levels are sparse and more than half of t...,Exclude raw one-hot names initially; test cros...,In-sample lookup performance is dominated by m...
